In [1]:
import numpy as np
import tensorflow as tf

2025-10-22 23:51:44.701584: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-22 23:51:44.701657: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-22 23:51:44.703357: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-22 23:51:44.712191: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-22 23:51:48.583383: W tensorflow/compiler/tf2

In [7]:
n_bits = 2
initial_range = (-1.0, 1.0)

L = 2 ** n_bits
lo, hi = initial_range
initial_levels = np.linspace(lo, hi, L, dtype=np.float32)
initial_levels

array([-1.        , -0.33333334,  0.33333334,  1.        ], dtype=float32)

In [8]:
np.diff(initial_levels)

array([0.6666666, 0.6666667, 0.6666666], dtype=float32)

In [24]:
threshold_offset = 80.0
initial_thresholds = np.array([400.0, 800.0, 2_000.0])
delta_thresholds = np.diff(initial_thresholds, prepend=threshold_offset)

print("Initial Thresholds:", initial_thresholds)
print("Delta Thresholds:", delta_thresholds)

Initial Thresholds: [ 400.  800. 2000.]
Delta Thresholds: [ 320.  400. 1200.]


In [39]:
def _param_delta(z, mode='softplus'):
    if mode == 'softplus':
        return tf.nn.softplus(z)
    else:
        return tf.where(z > 20.0, z, tf.math.expm1(z))        

def _inv_param_delta(z_positive, mode='softplus', z_thr=20.0):
    z = tf.convert_to_tensor(z_positive)       
    z = tf.cast(z, dtype=z.dtype)
    if mode == 'softplus':                           
        return tf.where(z > z_thr, z, tf.math.log(tf.math.expm1(z)))
    else: 
        return tf.math.log1p(z)

In [42]:
inv_param_delta_1 = _inv_param_delta(delta_thresholds, mode='other')
inv_param_delta_2 = _inv_param_delta(delta_thresholds, mode='softplus')

print(inv_param_delta_1)
print(inv_param_delta_2)

tf.Tensor([5.77144112 5.99396143 7.09090982], shape=(3,), dtype=float64)
tf.Tensor([ 320.  400. 1200.], shape=(3,), dtype=float64)


In [43]:
param_delta_1 = _param_delta(inv_param_delta_1, mode='other')
param_delta_2 = _param_delta(inv_param_delta_2, mode='softplus')

print(param_delta_1)
print(param_delta_2)

tf.Tensor([ 320.  400. 1200.], shape=(3,), dtype=float64)
tf.Tensor([ 320.  400. 1200.], shape=(3,), dtype=float64)
